### Зачем нужен tf-idf?



#### Пример поиска по запросу с редким словом

- **Коллекция документов** (три статьи о кошках):
  1. **«Абиссинская кошка: особенности породы»**  
     (слова: «абиссинская», «кошка», «порода», «уход» — часто)
  2. **«Как ухаживать за кошкой»**  
     (слова: «кошка», «уход», «кормление» — часто, но нет «абиссинская»)
  3. **«Чем кормить домашнюю кошку»**  
     (слова: «кошка», «корм», «миска» — часто, но нет «абиссинская» и «уход»)

- **Запрос пользователя**:  
  `абиссинская кошка уход`

- **Проблема**:
  - Слова «кошка» и «уход» встречаются во многих статьях → они **общие** (низкий IDF).
  - Слово **«абиссинская»** есть только в первой статье → оно **редкое** (высокий IDF).

- **Как сработает TF‑IDF**:
  - Статья 1: получает вес за все три слова, при этом вклад «абиссинская» будет большим из‑за редкости.
  - Статья 2: вес только от «кошка» и «уход» (их вклад мал, так как они общие).
  - Статья 3: вес только от «кошка» (ещё меньше).

- **Результат**:  
  На первом месте окажется статья **«Абиссинская кошка: особенности породы»**, потому что редкое слово точно указывает на неё, даже если общие слова есть везде.

### Как tf-idf решает проблему?

$$
\text{TF}(t, d) = \frac{\text{число вхождений } t \text{ в документ } d}{\text{общее число слов в документе } d}
$$

$$
\text{IDF}(t, D) = \log \left( \frac{\text{общее число документов}}{1 + \text{число документов, содержащих } t} \right)
$$

$$
\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)
$$



**Проблема**: слова «кошка», «уход», «идти» есть почти везде. Как сделать так, чтобы они не мешали, а редкие слова, наоборот, помогали?


**Формула состоит из двух частей**:

  **TF** — частота слова в документе  
  *Показывает, насколько важно слово для этого конкретного текста.*

  **IDF** — редкость слова во всех документах  
  *Показывает, насколько слово уникально для коллекции документов.*  
  Логарифм нужен, чтобы слишком редкие слова не «перевешивали» всё остальное.

**Итоговый вес слова**:  
  **TF‑IDF = TF × IDF**

**Как это решает проблему**:

Пусть у нас есть 100 статей про кошек:
- Слово кошка встречается в каждом тексте
- Слово абисинская встречается только в 1-м тексте

Возьмем 2 текста


  - Если слово частое (IDF маленький), даже большой TF не даст огромного веса.
  - Если слово редкое (IDF большой), но в документе оно встречается часто (TF большой), то вес будет максимальным.

- **Пример (коллекция из 100 статей про кошек)**:
  - Слово «кошка» есть в 95 статьях → IDF низкий.
  - Слово «абиссинская» есть только в 1 статье → IDF высокий.
  - В статье про абиссинскую кочку TF для «абиссинская» = 0.05, для «кошка» = 0.1
    - Вес «абиссинская» = 0.05 × большой_IDF  
    - Вес «кошка» = 0.1 × маленький_IDF  
    - Итог: слово «абиссинская» даст гораздо больший вклад, хотя встречается реже.



In [ ]:
import math
from typing import List, Dict, Tuple

def preprocess(text: str) -> List[str]:
    """Преобразует строку в нижний регистр и разбивает по пробелам."""
    return text.lower().split()

def build_vocab(documents: List[List[str]]) -> Dict[str, int]:
    """Создает словарь, сопоставляющий каждому уникальному слову индекс."""
    vocab = {}
    idx = 0
    for doc in documents:
        for word in doc:
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

def compute_idf(documents: List[List[str]], vocab: Dict[str, int]) -> Dict[str, float]:
    """
    Вычисляет обратную частоту документа для каждого слова в словаре.
    idf(слово) = log(всего_документов / (1 + частота_в_документах))
    """
    total_docs = len(documents)
    # Инициализируем частоты документов
    doc_freq = {word: 0 for word in vocab}

    for doc in documents:
        # Используем множество, чтобы учитывать слово только один раз на документ
        unique_words = set(doc)
        for word in unique_words:
            if word in doc_freq:
                doc_freq[word] += 1

    # Вычисляем idf
    idf = {}
    for word, freq in doc_freq.items():
        idf[word] = math.log(total_docs / (1 + freq))
    return idf

def vectorize(document: List[str],
              vocab: Dict[str, int],
              idf: Dict[str, float]) -> List[float]:
    """
    Преобразует документ (список слов) в вектор TF‑IDF.
    Вектор имеет ту же длину, что и словарь.
    """
    vec = [0.0] * len(vocab)

    # Подсчет частоты слов в этом документе
    tf_counts = {}
    for word in document:
        tf_counts[word] = tf_counts.get(word, 0) + 1

    for word, count in tf_counts.items():
        tf_counts[word] = count / len(document)

    # Заполняем вектор
    for word, count in tf_counts.items():
        if word in vocab:          # всегда должно быть истинно, но проверка для безопасности
            idx = vocab[word]
            vec[idx] = count * idf[word]

    return vec

def cosine_distance(vec1: List[float], vec2: List[float]) -> float:
    """
    Вычисляет косинусное расстояние = 1 - косинусное сходство.
    Возвращает 1.0, если норма любого вектора равна нулю.
    """
    dot = 0.0
    norm1 = 0.0
    norm2 = 0.0

    for i in range(len(vec1)):
        dot += vec1[i] * vec2[i]
        norm1 += vec1[i] * vec1[i]
        norm2 += vec2[i] * vec2[i]

    norm1 = math.sqrt(norm1)
    norm2 = math.sqrt(norm2)

    if norm1 == 0 or norm2 == 0:
        return 1.0

    similarity = dot / (norm1 * norm2)
    return 1 - similarity

def search(query: str,
           documents: List[List[str]],
           vocab: Dict[str, int],
           idf: Dict[str, float],
           doc_vectors: List[List[float]]) -> List[Tuple[List[str], float]]:
    """
    Обрабатывает запрос, сравнивает его со всеми векторами документов
    и возвращает результаты, отсортированные по возрастанию косинусного расстояния.
    """
    query_words = preprocess(query)
    query_vector = vectorize(query_words, vocab, idf)

    results = []
    for i, doc in enumerate(documents):
        dist = cosine_distance(query_vector, doc_vectors[i])
        results.append((doc, dist))

    results.sort(key=lambda x: x[1])   # сортировка по расстоянию, ближайшие первые
    return results

def main():
    # 10 документов о кошках, каждый список из 10‑15 слов в нижнем регистре
    documents = [
        ["кот", "сидел", "на", "коврике", "и", "спал", "часами"],
        ["кот", "играет", "с", "клубком", "пряжи", "на", "полу"],
        ["кот", "любит", "спать", "на", "солнечном", "месте", "у", "окна"],
        ["у", "моего", "кота", "мягкая", "шерсть", "и", "зеленые", "глаза", "которые", "светятся", "в", "темноте"],
        ["кошки", "независимые", "животные", "которые", "любят", "исследовать", "окрестности"],
        ["кот", "перепрыгнул", "через", "забор", "в", "сад", "чтобы", "погнаться", "за", "птицей"],
        ["кота", "по", "имени", "усатик", "любит", "есть", "рыбу", "и", "пить", "молоко", "каждый", "день"],
        ["кот", "поцарапал", "мебель", "своими", "когтями", "и", "оставил", "следы"],
        ["кошки", "часто", "мурлычут", "когда", "они", "счастливы", "и", "довольны", "жизнью"],
        ["кот", "гонялся", "за", "мышкой", "по", "дому", "пока", "она", "не", "убежала", "наружу"]
    ]

    # Построить словарь и IDF, затем векторизовать все документы один раз
    vocab = build_vocab(documents)
    idf = compute_idf(documents, vocab)
    doc_vectors = [vectorize(doc, vocab, idf) for doc in documents]

    # Получить запрос пользователя
    query = input("Введите ваш запрос: ")

    # Выполнить поиск
    results = search(query, documents, vocab, idf, doc_vectors)

    # Вывести результаты
    print("\nРезультаты поиска (ближайшие совпадения первыми):")
    for doc, dist in results:
        print(f"Расстояние: {dist:.4f}  ->  {' '.join(doc)}")

if __name__ == "__main__":
    main()

Введите ваш запрос: кот сидела на коврике

Результаты поиска (ближайшие совпадения первыми):
Расстояние: 0.4460  ->  кот сидел на коврике и спал часами
Расстояние: 0.8626  ->  кот играет с клубком пряжи на полу
Расстояние: 0.8641  ->  кот любит спать на солнечном месте у окна
Расстояние: 0.9831  ->  кот поцарапал мебель своими когтями и оставил следы
Расстояние: 0.9853  ->  кот перепрыгнул через забор в сад чтобы погнаться за птицей
Расстояние: 0.9862  ->  кот гонялся за мышкой по дому пока она не убежала наружу
Расстояние: 1.0000  ->  у моего кота мягкая шерсть и зеленые глаза которые светятся в темноте
Расстояние: 1.0000  ->  кошки независимые животные которые любят исследовать окрестности
Расстояние: 1.0000  ->  кота по имени усатик любит есть рыбу и пить молоко каждый день
Расстояние: 1.0000  ->  кошки часто мурлычут когда они счастливы и довольны жизнью
